# J5 — Vérification et synthèse

Objectif : vérifier les calculs NumPy avec PyTorch et rassembler les conclusions des expériences. PyTorch sert ici de référence ; le réseau et les optimiseurs du projet restent implémentés en NumPy.

## Travaux demandés
- [x] Comparer forward, BCE et gradients NumPy/PyTorch
- [x] Comparer une mise à jour SGD et Adam
- [x] Résumer les résultats et limites méthodologiques des cinq jalons


## G — Vérification avec PyTorch
Les deux modèles reçoivent les mêmes poids, biais et batch. On compare les prédictions, la BCE, les gradients puis les paramètres après une mise à jour. Le calcul est effectué en double précision pour limiter les écarts d’arrondi.

Cette cellule nécessite un kernel Jupyter où PyTorch est installé.


In [1]:
from data import SEED, generate_data, split_data, standardize_data
from model import init_params
from experiments import verify_pytorch_reference

X, y = generate_data(seed=SEED)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, seed=SEED)
X_train, X_val, X_test, mean, std = standardize_data(X_train, X_val, X_test)
params = init_params(seed=SEED)

errors = verify_pytorch_reference(
    X_train[:32], y_train[:32], params,
    sgd_rate=0.05, adam_rate=0.01,
)
for name, error in errors.items():
    print(f"{name}: {error:.3e}")


forward_max_abs_error: 1.110e-16
loss_abs_error: 0.000e+00
gradient_max_abs_error: 2.776e-17
sgd_parameter_max_abs_error: 5.551e-17
adam_parameter_max_abs_error: 6.939e-18


### Interprétation
Des écarts très faibles entre NumPy et PyTorch indiquent que le forward, la loss, le backward et les mises à jour comparées sont cohérents, à l’arrondi numérique près.

Résultats obtenus en double précision : erreur maximale du forward `1,110e-16`, écart de BCE `0`, erreur maximale des gradients `2,776e-17`, écart maximal après une mise à jour SGD `5,551e-17` et après une mise à jour Adam `6,939e-18`. Ces écarts sont compatibles avec les arrondis numériques ; les implémentations NumPy et PyTorch concordent sur ce batch.


## Synthèse des jalons
- J1 : le gradient checking donne une erreur relative `1,79e-11` ; sur 32 exemples, la loss baisse de `0,6726` à `0,1666`.
- J2 : selon la loss validation, les meilleurs taux testés sont SGD `0,05`, Momentum `0,1` et Adam `0,01` ; Momentum obtient `0,0825`.
- J3 : l’early stopping sélectionne l’époque 49 ; le modèle final obtient une accuracy test de `0,9867`. L2 et dropout n’améliorent pas ici la validation.
- J4 : une contrainte trop serrée dégrade la validation ; L2 et projection donnent des résultats proches à normes finales appariées.
- J5 : les résultats NumPy concordent avec PyTorch à la précision machine sur le batch vérifié.

### Limites méthodologiques
Un seul seed a été utilisé pour la plupart des comparaisons ; les résultats dépendent donc de ce split et de cette initialisation. La grille d’hyperparamètres reste limitée. Le jeu de test n’a été utilisé qu’après la sélection finale.
